In [ ]:
import pandas as pd

df = pd.read_csv("../data/IMDB Dataset.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df["sentiment"].value_counts()

In [ ]:
import re
import string
import nltk

from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

In [ ]:
df["review"][0]


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Tokenize
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)


In [ ]:
df["clean_review"] = df["review"].apply(clean_text)

In [ ]:
df.duplicated().sum()


In [ ]:
df = df.drop_duplicates()

In [ ]:
df["review_length"] = df["review"].str.len()

df["review_length"].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(df["review_length"], bins=30)
plt.title("Distribution of Review Length")
plt.xlabel("Review Length")
plt.ylabel("Count")
plt.show()

In [ ]:
df["sentiment"].value_counts().plot(kind="bar")
plt.title("Positive vs Negative Reviews")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
df["clean_review"] = df["review"].apply(clean_text)

In [ ]:
df[["review", "clean_review"]].head()

In [1]:
import pandas as pd
import numpy as np
import re
import string

from bs4 import BeautifulSoup

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prahladsingh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\prahladsingh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
df = pd.read_csv("../data/IMDB Dataset.csv")

In [3]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [4]:
df["clean_review"] = df["review"].apply(clean_text)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000
)

X = tfidf.fit_transform(df["clean_review"])

y = df["sentiment"]

In [6]:
X.shape

(50000, 5000)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [9]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [10]:
from sklearn.svm import LinearSVC

svm = LinearSVC()

svm.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [11]:
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": lr,
    "Naive Bayes": nb,
    "Linear SVM": svm
}

for name, model in models.items():
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(name, ":", round(acc,4))

Logistic Regression : 0.8869
Naive Bayes : 0.8521
Linear SVM : 0.8772


In [12]:
from sklearn.model_selection import GridSearchCV

params = {
    "C":[0.1,1,10]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

{'C': 1}
0.88195


In [13]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

best_model = grid.best_estimator_

pred = best_model.predict(X_test)

print(classification_report(y_test,pred))

print(confusion_matrix(y_test,pred))

              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      5000
    positive       0.88      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

[[4375  625]
 [ 506 4494]]


In [14]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test,pred)

0.8869

In [15]:
import joblib

joblib.dump(best_model,"../models/sentiment_model.pkl")

joblib.dump(tfidf,"../models/tfidf_vectorizer.pkl")

['../models/tfidf_vectorizer.pkl']

In [16]:
import os

os.listdir("../models")

['sentiment_model.pkl', 'tfidf_vectorizer.pkl']